# Evaluation Notebook
## AI Competitive Intelligence Copilot — Luxury Fashion

Evaluates three metrics from the spec:
1. **Relevance Precision@10** — top 10 retrieved items: % useful
2. **Event Classification F1** — model vs human-labeled event types
3. **Trend Precision@5** — top 5 trends: % judged valid

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from src.db import get_connection, get_latest_trends
from src.evaluation import precision_at_k, event_f1, trend_precision_at_k, EVENT_TYPES

## Load data from DB

In [ ]:
conn = get_connection()

# Load top 10 items by impact score
df_top10 = pd.read_sql_query("""
    SELECT i.item_id, i.competitor, i.source_name, i.title, i.source_url,
           e.event_type, e.impact_score, e.relevance_score, e.summary
    FROM items i
    JOIN events e ON i.item_id = e.item_id
    ORDER BY e.impact_score DESC
    LIMIT 10
""", conn)

# Load all events for F1 evaluation
df_all_events = pd.read_sql_query("""
    SELECT i.item_id, i.title, i.competitor, i.source_url,
           e.event_type, e.confidence_score, e.evidence_snippet, e.summary
    FROM events e
    JOIN items i ON e.item_id = i.item_id
    LIMIT 30
""", conn)

conn.close()
print(f"Loaded {len(df_top10)} top items and {len(df_all_events)} events for evaluation")

## Metric 1: Relevance Precision@10

Manually label each of the top 10 items as relevant (1) or not (0).

**Relevant** = the article genuinely discusses a competitive signal for Chanel, Dior, or Gucci.

Run the cell below, review the items, then fill in `human_labels`.

In [ ]:
# Display top 10 for manual review
pd.set_option('display.max_colwidth', 80)
df_top10[['competitor', 'event_type', 'impact_score', 'title', 'source_name']].style.background_gradient(subset=['impact_score'], cmap='RdYlGn')

In [ ]:
# ── FILL IN: 1 = relevant, 0 = not relevant (one per row above) ──
human_labels_p10 = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]  # edit this after reviewing

precision_at_10 = precision_at_k(human_labels_p10, k=10)
print(f"Relevance Precision@10: {precision_at_10:.0%}  ({sum(human_labels_p10)}/{len(human_labels_p10)} relevant)")

## Metric 2: Event Classification F1

Review a sample of events and provide the correct event type label.
Then compute F1 against the model's predictions.

In [ ]:
# Display events for labeling
for i, row in df_all_events.head(20).iterrows():
    print(f"[{i:02d}] {row['competitor']:8s} | Model: {row['event_type']:<35s} | {row['title'][:70]}")

In [ ]:
# Model predictions (from DB)
y_pred = df_all_events.head(20)['event_type'].tolist()

# ── FILL IN: your human labels for the 20 events above ──
y_true = y_pred.copy()  # replace with your annotations after reviewing

scores = event_f1(y_true, y_pred)
print(f"F1 Score (macro):    {scores['macro']:.3f}")
print(f"F1 Score (weighted): {scores['weighted']:.3f}")
print()
print(scores['report'])

f1_macro = scores['macro']

## Metric 3: Trend Precision@5

Review the top 5 detected trends and judge whether each is a valid strategic signal.

In [ ]:
trends = get_latest_trends(limit=5)
df_trends = pd.DataFrame(trends)

if df_trends.empty:
    print("No trends found. Run the pipeline first.")
else:
    display_cols = ['competitor', 'event_type', 'trend_score', 'count_7d', 'unique_sources', 'avg_impact', 'is_critical']
    df_trends[display_cols].style.background_gradient(subset=['trend_score'], cmap='Blues')

In [ ]:
# ── FILL IN: 1 = valid trend, 0 = false positive (one per row above) ──
human_trend_labels = [1, 1, 1, 1, 1]  # edit after reviewing

trend_precision_at_5 = trend_precision_at_k(human_trend_labels, k=5)
print(f"Trend Precision@5: {trend_precision_at_5:.0%}  ({sum(human_trend_labels[:5])}/{min(len(human_trend_labels), 5)} valid)")

## Summary Dashboard

In [ ]:
import matplotlib.pyplot as plt

metrics = {
    'Relevance\nPrecision@10': precision_at_10,
    'Event F1\n(macro)': f1_macro,
    'Trend\nPrecision@5': trend_precision_at_5,
}

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(metrics.keys(), metrics.values(), color=['#2c7bb6', '#d7191c', '#1a9641'], width=0.5)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Evaluation Metrics — AI Competitive Intelligence Copilot')
for bar, val in zip(bars, metrics.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.02, f'{val:.0%}', ha='center', fontweight='bold')
ax.axhline(0.7, color='gray', linestyle='--', linewidth=0.8, label='Target (70%)')
ax.legend()
plt.tight_layout()
plt.savefig('../data/evaluation_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to data/evaluation_metrics.png")

In [ ]:
# ── Step 3: Save model for pipeline use ───────────────────────────────────────
import joblib, os

model_path = os.path.join('..', 'data', 'event_classifier.joblib')
joblib.dump({"model": lr, "label_encoder": le}, model_path)
print(f"Saved classifier to {model_path}")
print(f"Classes: {list(le.classes_)}")
print(f"\nTo use in pipeline: set USE_TRAINED_CLASSIFIER=1")
print(f"  python -m src.pipeline --skip-ingest  (with env var set)")


In [ ]:
# ── Step 2: Train LogisticRegression on embeddings ────────────────────────────
import os
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from src.db import get_connection
from src.processing.embeddings import from_bytes
from src.evaluation import event_f1

labels_path = os.path.join('..', 'data', 'labeled_events.csv')
if not os.path.exists(labels_path):
    raise FileNotFoundError("Run the labeling cell first to generate labeled_events.csv")

df_labels = pd.read_csv(labels_path)
print(f"Loaded {len(df_labels)} labeled examples")

# Fetch embeddings for labeled items
conn = get_connection()
rows = conn.execute(
    f"SELECT i.item_id, i.embedding FROM items i "
    f"JOIN events e ON i.item_id = e.item_id "
    f"WHERE e.event_id IN ({','.join(['?']*len(df_labels))})",
    df_labels["event_id"].tolist(),
).fetchall()
conn.close()

item_emb = {r["item_id"]: from_bytes(r["embedding"]) for r in rows if r["embedding"]}
df_labels = df_labels[df_labels["item_id"].isin(item_emb)].reset_index(drop=True)

X = np.stack([item_emb[iid] for iid in df_labels["item_id"]])
y = df_labels["event_type"].values

le = LabelEncoder()
y_enc = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y_enc, test_size=0.2, random_state=42, stratify=y_enc if len(set(y_enc)) > 1 else None)

lr = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
lr.fit(X_train, y_train)

y_pred_enc = lr.predict(X_test)
y_pred_labels = le.inverse_transform(y_pred_enc)
y_true_labels = le.inverse_transform(y_test)

lr_scores = event_f1(list(y_true_labels), list(y_pred_labels))
print(f"\n── LogisticRegression (embeddings) ──")
print(f"F1 macro:    {lr_scores['macro']:.3f}")
print(f"F1 weighted: {lr_scores['weighted']:.3f}")

# Keyword baseline: predictions from DB are the "keyword" baseline
y_pred_kw = df_labels.loc[df_labels["item_id"].isin(item_emb), "event_type"].values
X_test_ids = df_labels.iloc[len(X_train):]["item_id"].values if len(X_train) < len(df_labels) else []

# Compare on same test split (use stored model predictions as keyword baseline)
conn2 = get_connection()
kw_rows = conn2.execute(
    "SELECT e.item_id, e.event_type FROM events e "
    "JOIN items i ON e.item_id = i.item_id"
).fetchall()
conn2.close()
kw_map = {r["item_id"]: r["event_type"] for r in kw_rows}

test_items = df_labels.iloc[len(X_train):]["item_id"].tolist()
kw_test_pred = [kw_map.get(iid, "collection_launch") for iid in test_items]
kw_scores = event_f1(list(y_true_labels), kw_test_pred)

print(f"\n── Keyword baseline ──")
print(f"F1 macro:    {kw_scores['macro']:.3f}")
print(f"F1 weighted: {kw_scores['weighted']:.3f}")

print(f"\n── Improvement ──")
print(f"Macro F1 delta: {lr_scores['macro'] - kw_scores['macro']:+.3f}")
print(f"\nFull report:\n{lr_scores['report']}")


In [ ]:
# ── FILL IN: correct any wrong model predictions ──────────────────────────────
# Format: {row_index: "correct_event_type"}
# Valid types: collection_launch, campaign_or_collaboration, pricing_or_exclusivity,
#              geographic_expansion, creative_direction, sustainability_or_sourcing,
#              celebrity_or_influencer_alignment, reputational_issue
#
# Example: corrections = {3: "reputational_issue", 17: "collection_launch"}

corrections = {}

# Apply corrections and save labeled CSV
df_labeled = df_label.copy()
for idx, correct_type in corrections.items():
    df_labeled.at[idx, "event_type"] = correct_type

import os
labels_path = os.path.join('..', 'data', 'labeled_events.csv')
df_labeled[["event_id", "item_id", "event_type"]].to_csv(labels_path, index=False)
print(f"Saved {len(df_labeled)} labeled events to {labels_path}")
print(f"Corrections applied: {len(corrections)}")
print(f"\nLabel distribution:\n{df_labeled['event_type'].value_counts().to_string()}")


In [ ]:
# ── Step 1: Load events for labeling ──────────────────────────────────────────
# Shows model predictions; edit `corrections` to fix wrong labels.

conn = get_connection()
df_label = pd.read_sql_query("""
    SELECT e.event_id, e.item_id, i.competitor, i.title, e.event_type,
           e.confidence_score, e.evidence_snippet
    FROM events e
    JOIN items i ON e.item_id = i.item_id
    ORDER BY e.created_at DESC
    LIMIT 150
""", conn)
conn.close()

print(f"Loaded {len(df_label)} events for labeling\n")
print(f"{'#':<4} {'Brand':<8} {'Model prediction':<38} {'Title'[:55]}")
print("-" * 100)
for i, row in df_label.iterrows():
    print(f"{i:<4} {row['competitor']:<8} {row['event_type']:<38} {str(row['title'])[:55]}")


## Classifier Fine-Tuning

Train a LogisticRegression classifier on top of the stored 384-dim embeddings.
Steps:
1. **Label** — review model predictions and correct any wrong event types
2. **Train** — fit LogisticRegression on labeled embeddings, compare F1 vs keyword baseline
3. **Save** — persist model to `data/event_classifier.joblib` for pipeline use